# Phase 1: Circuit Tracing — Triangle Inequality

This notebook implements the first experimental phase of the thesis project. Its goal is to apply the `circuit-tracer` attribution graph methodology to a controlled set of mathematical true/false statements about the **triangle inequality**, in order to identify and analyse the internal features of the Gemma-2-2b language model that influence its classification of these statements.

The experiment follows five sequential steps:

1. **Baseline sweep** — record the model's unaided probability for `True` vs `False` across all questions and prompt templates.
2. **Attribution graph construction** — for each question, build a graph showing which internal features drove the `True`/`False` decision.
3. **Top feature extraction** — identify the most influential active features per question.
4. **Cross-question consistency** — check whether the same features recur across multiple questions, pointing to shared circuits.
5. **Circuit interventions** — amplify and ablate recurring features to test whether they causally influence the model's output.

All questions and prompt templates are defined in `my_work/data/questions.json` and loaded at runtime. To change, add, or remove questions, edit that file — no changes to this notebook are required.

---

## Section 0 — Environment Setup

This section handles all environment configuration tasks that must run before any machine-learning code. It is adapted from the tested setup in `my_work/notebooks/runpod_baseline.ipynb`.

There are three cells in this section:

1. **Dependency bootstrap** — installs the `circuit-tracer` package into the current kernel. This is a one-time step on a fresh pod or new kernel. Run it once and then restart the kernel before continuing.
2. **Runtime and Hugging Face setup** — locates the repository root on disk, validates that all required Python packages are installed, pins the Hugging Face package stack to compatible versions, loads the `.env` file for local runs, and configures persistent cache directories when running on Runpod.
3. **Sanity check** — prints GPU availability, torch version, and all relevant environment variables so you can confirm the environment is correct before running expensive model loading.

In [1]:
# Run this cell once on a fresh pod or new kernel, then restart the kernel.
# After restarting, skip this cell and continue from the next one.

# %pip install --no-cache-dir -e /workspace/thesis_circuit_breaker


In [1]:
#@title Runtime + Hugging Face Setup (Local/Runpod) { display-mode: "form" }

import os
import sys
from pathlib import Path

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

print("Configuring local/Runpod environment...")


def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None


_root = _find_repo_root()
if _root is not None:
    root_str = str(_root)
    if root_str not in sys.path:
        sys.path.insert(0, root_str)
    print(f"Repo root on sys.path: {_root}")

    import importlib.util
    import subprocess

    required_modules = ["safetensors", "transformers", "tokenizers", "nnsight", "transformer_lens"]
    missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
    if missing:
        raise RuntimeError(
            f"Missing dependencies: {missing}. "
            f"Install with: `pip install -e {_root}` then restart kernel."
        )

    from importlib.metadata import version as _pkg_version

    def _parse(v):
        core = v.split("+", 1)[0].strip()
        parts = core.split(".")
        nums = []
        for p in parts[:3]:
            digits = "".join(ch for ch in p if ch.isdigit())
            nums.append(int(digits) if digits else 0)
        while len(nums) < 3:
            nums.append(0)
        return tuple(nums)

    def _lt(a, b): return _parse(a) < _parse(b)
    def _gt(a, b): return _parse(a) > _parse(b)
    def _gte(v, minimum): return not _lt(v, minimum)

    def _pin_hf_stack():
        try:
            hf_hub_v = _pkg_version("huggingface_hub")
        except Exception:
            hf_hub_v = ""
        try:
            tf_v = _pkg_version("transformers")
        except Exception:
            tf_v = ""
        needs_pin = (hf_hub_v and _gte(hf_hub_v, "1.0.0")) or (
            tf_v and (_lt(tf_v, "4.56.0") or _gt(tf_v, "4.57.3"))
        )
        if needs_pin:
            print("Pinning huggingface_hub + transformers to repo-compatible versions...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                "huggingface_hub<1.0.0", "transformers>=4.56.0,<=4.57.3",
            ])
            raise RuntimeError(
                "Pinned huggingface_hub/transformers. Restart the kernel, then rerun from the top."
            )

    _pin_hf_stack()
else:
    print("Could not locate circuit_tracer repo. Set CT_REPO_DIR or clone into /workspace.")

try:
    from typing_extensions import TypeIs  # type: ignore[attr-defined]
except Exception:
    print("typing_extensions too old; upgrading...")
    %pip install -q --no-cache-dir "typing_extensions>=4.12.2"
    raise RuntimeError("Installed typing_extensions>=4.12.2. Restart kernel and rerun.")

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv is not None and _root is not None:
    _env_file = _root / ".env"
    if _env_file.is_file():
        load_dotenv(_env_file)
        print(f"Loaded {_env_file}")

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if hf_token and not os.environ.get("HUGGINGFACE_HUB_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    cache_dirs = {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
        "HF_DATASETS_CACHE": hf_home / "datasets",
        "TORCH_HOME": persistent_root / "torch",
        "CT_OUTPUT_DIR": persistent_root / "results" / "phase1",
    }
    for key, path in cache_dirs.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)
    print(f"Runpod persistent storage configured under {persistent_root}")

from huggingface_hub import get_token

if get_token() is None:
    print(
        "No Hugging Face token found. google/gemma-2-2b is a gated model. "
        "Accept the licence at https://huggingface.co/google/gemma-2-2b and set HF_TOKEN."
    )
else:
    print("Hugging Face token detected.")


Configuring local/Runpod environment...
Repo root on sys.path: /workspace/thesis_circuit_breaker
Runpod persistent storage configured under /workspace
Hugging Face token detected.


In [2]:
import os
import torch

print(f'torch version : {torch.__version__}')
print(f'cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu           : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: no GPU detected — inference will be very slow on CPU.')

for key in [
    'HF_HOME', 'HUGGINGFACE_HUB_CACHE', 'TRANSFORMERS_CACHE',
    'HF_DATASETS_CACHE', 'TORCH_HOME', 'CT_OUTPUT_DIR',
]:
    print(f'{key}={os.environ.get(key, "<unset>")}')

print('HF token set:', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')))


torch version : 2.4.1+cu124
cuda available: True
gpu           : NVIDIA RTX A4500
HF_HOME=/workspace/hf
HUGGINGFACE_HUB_CACHE=/workspace/hf/hub
TRANSFORMERS_CACHE=/workspace/hf/transformers
HF_DATASETS_CACHE=/workspace/hf/datasets
TORCH_HOME=/workspace/torch
CT_OUTPUT_DIR=/workspace/results/phase1
HF token set: True


---

## Imports

All Python imports used throughout this notebook are consolidated here into a single cell. Running this cell after environment setup ensures that every module is available before any experiment code runs. If an import fails at this stage, it is a signal that the dependency installation step above did not complete successfully.

In [3]:
import csv
import json
import sys
from collections import defaultdict
from pathlib import Path

import torch
from IPython.display import IFrame, Markdown, display

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.attribution.targets import CustomTarget
from circuit_tracer.frontend.local_server import serve
from circuit_tracer.utils import create_graph_files
from circuit_tracer.utils import get_default_device
from circuit_tracer.utils.demo_utils import (
    cleanup_cuda,
    display_ablation_chart,
    display_token_probs,
    display_top_features_comparison,
    get_top_features,
    get_unembed_vecs,
)

print('All imports successful.')


/workspace/venvs/ct/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


All imports successful.


---

## Question Configuration

The question bank and prompt templates are stored in `my_work/data/questions.json`, which is the **single source of truth** for this experiment. Nothing about the questions is hardcoded in this notebook — all content comes from that file at runtime.

To modify, add, or remove questions later (for example, to extend to Pythagorean theorem or other geometry topics), simply edit `questions.json` and re-run this cell. No notebook changes are needed.

The config defines:

- **`templates`** — three prompt wrapper formats (`T1`, `T2`, `T3`) that differ in phrasing.
- **`active_template`** — which template is used for attribution and intervention runs.
- **`questions`** — the list of statements, each with a ground-truth label and notes.

The baseline sweep runs all questions against all three templates. Attribution and intervention runs use only the active template to keep compute costs manageable.

In [4]:
# Read configuration from an existing questions.json file.
# This cell does not auto-create files; it only resolves and loads paths.
def _find_config_path() -> Path:
    candidates = []

    # Best option: _root resolved by env-setup cell
    if '_root' in globals() and _root is not None:  # type: ignore[name-defined]
        candidates.append(_root / 'my_work' / 'data' / 'questions.json')  # type: ignore[name-defined]

    # CT_REPO_DIR env var override
    repo_override = os.environ.get('CT_REPO_DIR')
    if repo_override:
        candidates.append(Path(repo_override).expanduser().resolve() / 'my_work' / 'data' / 'questions.json')

    # CWD walk
    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        candidates.extend([
            parent / 'my_work' / 'data' / 'questions.json',
            parent / 'data' / 'questions.json',
        ])

    # Common workspace roots (Runpod-like)
    for root in [Path('/workspace'), Path('/root'), Path('/home')]:
        if not root.is_dir():
            continue
        candidates.append(root / 'my_work' / 'data' / 'questions.json')
        try:
            for child in root.iterdir():
                if child.is_dir():
                    candidates.append(child / 'my_work' / 'data' / 'questions.json')
                    candidates.append(child / 'data' / 'questions.json')
        except Exception:
            pass

    seen = set()
    deduped = []
    for cand in candidates:
        if cand in seen:
            continue
        seen.add(cand)
        deduped.append(cand)
        if cand.is_file():
            return cand

    top_checked = '\n'.join(f'  - {c}' for c in deduped[:12])
    raise FileNotFoundError(
        'Could not locate my_work/data/questions.json. '
        f'Current working directory is: {cwd}. '
        'Set CT_REPO_DIR to your repository path and ensure the file exists on this machine. '
        'Checked paths (first 12):\n' + top_checked
    )


CONFIG_PATH = _find_config_path()
cfg = json.loads(CONFIG_PATH.read_text())
TEMPLATES = cfg['templates']
QUESTIONS = cfg['questions']
ACTIVE_TEMPLATE = cfg['metadata']['active_template']

NOTEBOOK_DIR = CONFIG_PATH.parent.parent / 'notebooks'

print(f'Current working dir: {Path.cwd().resolve()}')
print(f'Config loaded from : {CONFIG_PATH}')
print(f'Templates defined  : {list(TEMPLATES.keys())}')
print(f'Active template    : {ACTIVE_TEMPLATE}')
print(f'Questions loaded   : {len(QUESTIONS)}')
print()
for q in QUESTIONS:
    label_str = 'TRUE ' if q['label'] else 'FALSE'
    print(f"  [{label_str}] {q['id']}: {q['statement'][:80]}")


Current working dir: /
Config loaded from : /workspace/thesis_circuit_breaker/my_work/data/questions.json
Templates defined  : ['T1', 'T2', 'T3']
Active template    : T1
Questions loaded   : 8

  [TRUE ] TI-01: For any triangle, the sum of any two sides is greater than the third side.
  [FALSE] TI-02: For any triangle, the sum of any two sides is less than the third side.
  [TRUE ] TI-03: For any triangle, the sum of any two sides is greater than or equal to the third
  [FALSE] TI-04: In a triangle, it is possible for one side to be longer than the sum of the othe
  [TRUE ] TI-05: In every triangle, each side must be shorter than the combined length of the oth
  [FALSE] TI-06: For any triangle with sides a, b, and c, the inequality a + b ≤ c can hold.
  [TRUE ] TI-07: Three line segments can form a triangle only if each segment is less than the su
  [FALSE] TI-08: The triangle inequality states that the sum of two sides of a triangle is always


In [5]:
print("CONFIG_PATH:", CONFIG_PATH)
raw = CONFIG_PATH.read_text()
print(raw[:300])

CONFIG_PATH: /workspace/thesis_circuit_breaker/my_work/data/questions.json
{
  "metadata": {
    "version": "1.0",
    "description": "Phase 1 pilot question bank — triangle inequality. Edit this file to add, remove, or replace questions. The notebook loads it at runtime, so no notebook changes are needed when updating the question set.",
    "active_template": "T1"
  },
 


---

## Model Loading

We use `ReplacementModel.from_pretrained`, which loads the base language model together with its pre-trained cross-layer transcoder (CLT) weights. The replacement model is the version of Gemma-2-2b in which MLP outputs are approximated by sparse transcoder features. This is the model that attribution graphs are built on, not the raw base model.

A few configuration notes:

- **`MODEL_NAME`** — the Hugging Face model ID. `google/gemma-2-2b` is the base (non-instruct) model.
- **`TRANSCODER_NAME`** — the shortcut `"gemma"` resolves to the GemmaScope transcoder weights hosted on Hugging Face.
- **`dtype=torch.bfloat16`** — loads weights in 16-bit brain float, halving memory usage while preserving numeric stability.
- **`BACKEND="transformerlens"`** — uses the TransformerLens backend, which is faster and more memory-efficient than the nnsight backend for Gemma.

The first run will download and cache model weights; subsequent runs reuse the cache. Weights are stored under the `HF_HOME` path configured in Section 0.

In [6]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.mem_get_info())

(20955660288, 21151088640)


In [7]:
MODEL_NAME = 'google/gemma-2-2b'
TRANSCODER_NAME = 'gemma'
BACKEND = 'transformerlens'

device = get_default_device()
print(f'Loading model on {device} (cuda > mps > cpu)')

model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=torch.bfloat16,
    backend=BACKEND,
    device=device,
)

print(f'Model loaded: {MODEL_NAME}')


Loading model on cuda (cuda > mps > cpu)


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
Model loaded: google/gemma-2-2b


---

## Token Sanity Check

Before running any attribution, we must confirm that `True` and `False` each tokenise to **exactly one token** in Gemma's vocabulary. This matters because the attribution target is defined as the direction `logit(True) - logit(False)` in the model's output space. If either word maps to multiple tokens (or to an unexpected token ID), the target vector will be wrong and all downstream attribution results will be misleading.

We check both `' True'` and `' False'` with a leading space, because at the answer position in the prompt the model is expected to output a space-prefixed word. This corresponds to the Gemma tokeniser's convention of using a special marker for word-initial tokens.

The cell will raise an `AssertionError` if either token is not a single token, preventing the experiment from proceeding silently with incorrect targets.

In [8]:
TOKEN_TRUE_STR = ' True'
TOKEN_FALSE_STR = ' False'

tokenizer = model.tokenizer

ids_true = tokenizer.encode(TOKEN_TRUE_STR, add_special_tokens=False)
ids_false = tokenizer.encode(TOKEN_FALSE_STR, add_special_tokens=False)

assert len(ids_true) == 1, (
    f"'{TOKEN_TRUE_STR}' tokenises to {len(ids_true)} tokens: {ids_true}. "
    'Adjust TOKEN_TRUE_STR so it maps to exactly one token.'
)
assert len(ids_false) == 1, (
    f"'{TOKEN_FALSE_STR}' tokenises to {len(ids_false)} tokens: {ids_false}. "
    'Adjust TOKEN_FALSE_STR so it maps to exactly one token.'
)

IDX_TRUE = ids_true[0]
IDX_FALSE = ids_false[0]

print(f"Token '{TOKEN_TRUE_STR}'  -> vocab ID {IDX_TRUE}  (decoded: '{tokenizer.decode([IDX_TRUE])}')") 
print(f"Token '{TOKEN_FALSE_STR}' -> vocab ID {IDX_FALSE} (decoded: '{tokenizer.decode([IDX_FALSE])}')") 
print()
print('Sanity check passed. Token IDs are valid for attribution target construction.')


Token ' True'  -> vocab ID 5569  (decoded: ' True')
Token ' False' -> vocab ID 7662 (decoded: ' False')

Sanity check passed. Token IDs are valid for attribution target construction.


---

## Section 1 — Baseline Sweep

The baseline sweep measures the model's unaided next-token preference for `True` vs `False` on each question, across all three prompt templates. No attribution or intervention is performed at this stage.

For each `(question, template)` combination we:

1. Format the full prompt by inserting the statement into the template.
2. Run a forward pass of the replacement model.
3. Read the raw logits at the final token position.
4. Record `logit(True)`, `logit(False)`, and their difference (`margin`).

A positive margin means the model predicts `True`; a negative margin means it predicts `False`. The magnitude of the margin indicates how confident the prediction is.

Results are saved to `results/phase1/baseline_results.csv` so they are not lost if the kernel is interrupted later. The file is overwritten on each full run.

In [9]:
def format_prompt(question: dict, template_key: str) -> str:
    """Insert a question statement into a named prompt template."""
    return TEMPLATES[template_key].replace('{S}', question['statement'])


In [10]:
RESULTS_DIR = NOTEBOOK_DIR.parent / 'results' / 'phase1'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_records = []

for q in QUESTIONS:
    for tkey in TEMPLATES:
        prompt = format_prompt(q, tkey)
        input_ids = model.ensure_tokenized(prompt)

        with torch.no_grad():
            logits, _ = model.get_activations(input_ids)

        last = logits.squeeze(0)[-1]
        l_true = last[IDX_TRUE].item()
        l_false = last[IDX_FALSE].item()
        margin = l_true - l_false
        predicted = margin > 0
        correct = predicted == q['label']

        baseline_records.append({
            'id': q['id'],
            'template': tkey,
            'label': q['label'],
            'logit_true': round(l_true, 4),
            'logit_false': round(l_false, 4),
            'margin': round(margin, 4),
            'predicted': predicted,
            'correct': correct,
        })

        cleanup_cuda()

csv_path = RESULTS_DIR / 'baseline_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(baseline_records[0].keys()))
    writer.writeheader()
    writer.writerows(baseline_records)

print(f'Baseline sweep complete. {len(baseline_records)} runs saved to {csv_path}')


Baseline sweep complete. 24 runs saved to /workspace/thesis_circuit_breaker/my_work/results/phase1/baseline_results.csv


In [11]:
print(f"{'ID':<8} {'Tmpl':<5} {'Label':<6} {'Margin':>8} {'Pred':<6} {'OK'}")
print('-' * 46)
for r in baseline_records:
    label_str = 'True ' if r['label'] else 'False'
    pred_str  = 'True ' if r['predicted'] else 'False'
    ok_str    = 'OK' if r['correct'] else 'WRONG'
    print(f"{r['id']:<8} {r['template']:<5} {label_str:<6} {r['margin']:>8.4f} {pred_str:<6} {ok_str}")

print()
for tkey in TEMPLATES:
    recs = [r for r in baseline_records if r['template'] == tkey]
    n_correct = sum(1 for r in recs if r['correct'])
    print(f'Accuracy {tkey}: {n_correct}/{len(recs)} ({100 * n_correct / len(recs):.0f}%)')


ID       Tmpl  Label    Margin Pred   OK
----------------------------------------------
TI-01    T1    True     2.0000 True   OK
TI-01    T2    True     2.0000 True   OK
TI-01    T3    True     1.5000 True   OK
TI-02    T1    False    1.8750 True   WRONG
TI-02    T2    False    2.1250 True   WRONG
TI-02    T3    False    1.3750 True   WRONG
TI-03    T1    True     2.0000 True   OK
TI-03    T2    True     2.1250 True   OK
TI-03    T3    True     1.5000 True   OK
TI-04    T1    False    2.0000 True   WRONG
TI-04    T2    False    2.3750 True   WRONG
TI-04    T3    False    1.3750 True   WRONG
TI-05    T1    True     1.7500 True   OK
TI-05    T2    True     2.0000 True   OK
TI-05    T3    True     1.5000 True   OK
TI-06    T1    False    1.7500 True   WRONG
TI-06    T2    False    2.2500 True   WRONG
TI-06    T3    False    1.5000 True   WRONG
TI-07    T1    True     2.0000 True   OK
TI-07    T2    True     1.8750 True   OK
TI-07    T3    True     1.3750 True   OK
TI-08    T1    False    

---

## Section 2 — Attribution Graph Construction

An attribution graph describes, for a specific prompt and a chosen output direction, which internal model components (features, token embeddings, error nodes) contributed to the model's preference and by how much. The graph is computed by the replacement model, which uses sparse transcoder features instead of raw MLP neurons.

### Attribution target: `logit(True) - logit(False)`

Rather than attributing to a single output token, we attribute to the **direction** `logit(True) - logit(False)` in the model's output space. This is a `CustomTarget` constructed as the difference of the two unembedding vectors, weighted by the absolute difference in their softmax probabilities.

Using a difference direction has a key advantage: features that contribute positively are specifically pushing the model toward `True` *over* `False`, not merely increasing output volume in general. This makes the attribution more diagnostic for understanding the classification decision.

### Compute cost and graph storage

Attribution runs are the most expensive step in this notebook. Each run processes up to `max_feature_nodes=8192` active transcoder features in batches. `offload='disk'` reduces GPU memory pressure by temporarily moving parts of the computation to disk. Graphs are saved as `.pt` files so they can be reloaded for later analysis without re-running attribution.

In [12]:
def build_true_false_target(
    model, prompt: str, idx_true: int, idx_false: int, backend: str
) -> CustomTarget:
    """Build a CustomTarget for the direction logit(True) - logit(False).

    The attribution vector is the difference of the unembedding columns for the two
    answer tokens. The weight is the absolute probability gap between them, floored
    at 1e-6 to avoid a zero-weight target when probabilities are almost equal.
    """
    vec_true, vec_false = get_unembed_vecs(model, [idx_true, idx_false], backend)
    diff_vec = vec_true - vec_false

    input_ids = model.ensure_tokenized(prompt)
    with torch.no_grad():
        logits, _ = model.get_activations(input_ids)
    probs = torch.softmax(logits.squeeze(0)[-1].float(), dim=-1)
    diff_prob = max((probs[idx_true] - probs[idx_false]).abs().item(), 1e-6)

    return CustomTarget(
        token_str='logit(True)-logit(False)',
        prob=diff_prob,
        vec=diff_vec,
    )


In [13]:
ATTR_KWARGS = dict(batch_size=256, max_feature_nodes=8192, offload='disk', verbose=True)

GRAPH_DIR = RESULTS_DIR / 'graphs'
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

graphs = {}

for q in QUESTIONS:
    qid = q['id']
    prompt = format_prompt(q, ACTIVE_TEMPLATE)
    target = build_true_false_target(model, prompt, IDX_TRUE, IDX_FALSE, backend=BACKEND)

    print(f"\n--- Attributing {qid} (label={q['label']}) ---")
    graph = attribute(
        prompt=prompt,
        model=model,
        attribution_targets=[target],
        **ATTR_KWARGS,
    )

    graphs[qid] = graph
    graph_path = GRAPH_DIR / f'{qid}_{ACTIVE_TEMPLATE}.pt'
    graph.to_pt(graph_path)

    print(f'{qid}: {graph.active_features.shape[0]} active features -> saved {graph_path.name}')
    cleanup_cuda()

print(f'\nAttribution complete. {len(graphs)} graphs saved to {GRAPH_DIR}')


Phase 0: Precomputing activations and vectors



--- Attributing TI-01 (label=True) ---


Precomputation completed in 4.55s
Found 34345 active features
Phase 1: Running forward pass
Forward pass completed in 0.58s
Phase 2: Building input vectors
Using 1 custom attribution targets with total weight 0.0055
Will include 8192 of 34345 feature nodes
Input vectors built in 5.38s
Phase 3: Computing logit attributions
Logit attributions completed in 0.86s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:13<00:00, 587.61it/s]
Feature attributions completed in 13.94s
Attribution completed in 30.69s
Phase 0: Precomputing activations and vectors


TI-01: 34345 active features -> saved TI-01_T1.pt

--- Attributing TI-02 (label=False) ---


Precomputation completed in 0.92s
Found 34120 active features
Phase 1: Running forward pass
Forward pass completed in 0.55s
Phase 2: Building input vectors
Using 1 custom attribution targets with total weight 0.0054
Will include 8192 of 34120 feature nodes
Input vectors built in 5.25s
Phase 3: Computing logit attributions
Logit attributions completed in 0.41s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:13<00:00, 603.65it/s]
Feature attributions completed in 13.57s
Attribution completed in 28.76s
Phase 0: Precomputing activations and vectors


...

TI-06: 38382 active features -> saved TI-06_T1.pt

--- Attributing TI-07 (label=True) ---


Precomputation completed in 1.13s
Found 39104 active features
Phase 1: Running forward pass
Forward pass completed in 0.63s
Phase 2: Building input vectors
Using 1 custom attribution targets with total weight 0.0048
Will include 8192 of 39104 feature nodes
Input vectors built in 5.36s
Phase 3: Computing logit attributions
Logit attributions completed in 0.66s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:15<00:00, 515.72it/s]
Feature attributions completed in 15.89s
Attribution completed in 30.22s
Phase 0: Precomputing activations and vectors


TI-07: 39104 active features -> saved TI-07_T1.pt

--- Attributing TI-08 (label=False) ---


Precomputation completed in 1.03s
Found 36934 active features
Phase 1: Running forward pass
Forward pass completed in 0.61s
Phase 2: Building input vectors
Using 1 custom attribution targets with total weight 0.0081
Will include 8192 of 36934 feature nodes
Input vectors built in 5.40s
Phase 3: Computing logit attributions
Logit attributions completed in 0.46s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:14<00:00, 548.64it/s]
Feature attributions completed in 14.93s
Attribution completed in 30.63s


TI-08: 36934 active features -> saved TI-08_T1.pt

Attribution complete. 8 graphs saved to /workspace/thesis_circuit_breaker/my_work/results/phase1/graphs


In [ ]:
features, scores = get_top_features(graphs["TI-01"], n=20)
print("target weight:", graphs["TI-01"].logit_probabilities)
print("max abs score:", max(abs(s) for s in scores))
print("scores sci:", [f"{s:.3e}" for s in scores[:10]])

---

## Section 3 — Top Feature Extraction

With the attribution graphs computed, we now extract the most influential features from each graph. `get_top_features` ranks all active features by their total multi-hop influence on the attribution target, meaning it accounts not just for direct effects on the output, but also for indirect effects mediated through chains of other features.

We extract the top 20 features per question. For each feature, the result is a tuple `(layer, position, feature_index)` that uniquely identifies it in the transcoder dictionary, along with a scalar influence score.

The side-by-side display includes Neuronpedia links so you can click through to see what text examples activate each feature most strongly. This is the primary tool for forming provisional interpretability labels such as 'rule-content', 'lexical cue', 'format/instruction', or 'unclear'.

A positive influence score means the feature pushes the model toward `True` over `False`. A negative score means it pushes toward `False`.

In [14]:
TOP_N = 20

top_features_by_question = {}
top_scores_by_question = {}

for qid, graph in graphs.items():
    features, scores = get_top_features(graph, n=TOP_N)
    top_features_by_question[qid] = features
    top_scores_by_question[qid] = scores

display_top_features_comparison(
    feature_sets=top_features_by_question,
    scores_sets=top_scores_by_question,
    neuronpedia_model='gemma-2-2b',
)


#,Node,Score
1,"(0, 2, 16200)",0.0000
2,"(0, 31, 6051)",0.0000
3,"(24, 31, 4640)",0.0000
4,"(1, 28, 14349)",0.0000
5,"(0, 1, 16229)",0.0000
6,"(1, 1, 11660)",0.0000
7,"(16, 31, 9223)",0.0000
8,"(0, 1, 6083)",0.0000
9,"(2, 30, 13635)",0.0000
10,"(21, 31, 2997)",0.0000


---

## Section 4 — Cross-Question Feature Consistency

A single feature appearing as a top contributor in one question could be coincidental. A feature that consistently appears in the top-20 across multiple questions is a much stronger candidate for being part of a genuine, reusable circuit.

This section computes, for each unique feature ID that appears in any top-20 list, how many of the eight questions it appears in. Features that appear in three or more questions are considered **recurring** and are promoted to intervention candidates in the next section.

The frequency count is a coarse but effective first filter. More rigorous consistency analysis — such as checking whether a feature appears specifically for true statements versus false ones — is supported by the printed table and left to manual inspection.

In [15]:
feature_frequency: dict = defaultdict(list)

for qid, features in top_features_by_question.items():
    for feat in features:
        feature_frequency[tuple(feat)].append(qid)

sorted_features = sorted(
    feature_frequency.items(), key=lambda x: len(x[1]), reverse=True
)

print(f'Unique features across all top-{TOP_N} lists: {len(sorted_features)}')
print()
print(f"{'Feature (layer, pos, feat_idx)':<36} {'Freq':>5}  Questions")
print('-' * 75)
for feat, qids in sorted_features:
    if len(qids) < 2:
        break
    layer, pos, feat_idx = feat
    print(f'({layer:>3}, {pos:>3}, {feat_idx:>8})                 {len(qids):>4}  {", ".join(qids)}')


Unique features across all top-20 lists: 68

Feature (layer, pos, feat_idx)        Freq  Questions
---------------------------------------------------------------------------
(  0,   2,    16200)                    8  TI-01, TI-02, TI-03, TI-04, TI-05, TI-06, TI-07, TI-08
(  0,   1,    16229)                    8  TI-01, TI-02, TI-03, TI-04, TI-05, TI-06, TI-07, TI-08
(  1,   1,    11660)                    8  TI-01, TI-02, TI-03, TI-04, TI-05, TI-06, TI-07, TI-08
(  0,   1,     6083)                    8  TI-01, TI-02, TI-03, TI-04, TI-05, TI-06, TI-07, TI-08
(  0,   1,     6421)                    8  TI-01, TI-02, TI-03, TI-04, TI-05, TI-06, TI-07, TI-08
(  0,   1,    11167)                    8  TI-01, TI-02, TI-03, TI-04, TI-05, TI-06, TI-07, TI-08
(  0,   1,     6381)                    7  TI-01, TI-02, TI-03, TI-04, TI-05, TI-07, TI-08
(  0,   1,    10345)                    7  TI-01, TI-02, TI-03, TI-05, TI-06, TI-07, TI-08
(  0,   1,     8489)                    5  TI-01, TI-02

---

## Section 5 — Circuit Interventions

Attribution graphs show correlation between feature activity and output preferences. To make a stronger claim that a feature is *causally* involved in the classification decision, we must perturb it and observe whether the output changes in the predicted direction.

For each question, we apply two types of intervention to the top recurring features identified in Section 4:

- **Amplify (5x)** — scale each selected feature's activation to five times its natural value. If the features are pushing toward `True`, amplifying them should increase the `True-False` margin.
- **Ablate (0x)** — set each selected feature's activation to zero, removing its contribution entirely. If the features were supporting `True`, ablation should decrease the margin or flip the prediction.

We record `margin_before`, `margin_after`, and `delta_margin` for each combination. A consistent, directionally correct `delta_margin` across multiple questions is evidence that the recurring features form a real, interpretable circuit.

Only features that are actually **active on a given prompt** can be intervened on. A recurring feature may not be active on every question, so the effective intervention set is filtered per prompt. The `n_features` column records how many candidates were actually active for each item.

In [16]:
RECURRING_THRESHOLD = 3
TOP_K_INTERV = 10

recurring_features = [
    feat for feat, qids in sorted_features if len(qids) >= RECURRING_THRESHOLD
][:TOP_K_INTERV]

print(f'Recurring features (freq >= {RECURRING_THRESHOLD}): {len(recurring_features)}')
for f in recurring_features:
    layer, pos, feat_idx = f
    freq = len(feature_frequency[f])
    print(f'  ({layer}, {pos}, {feat_idx}) — appears in {freq} questions')


Recurring features (freq >= 3): 10
  (0, 2, 16200) — appears in 8 questions
  (0, 1, 16229) — appears in 8 questions
  (1, 1, 11660) — appears in 8 questions
  (0, 1, 6083) — appears in 8 questions
  (0, 1, 6421) — appears in 8 questions
  (0, 1, 11167) — appears in 8 questions
  (0, 1, 6381) — appears in 7 questions
  (0, 1, 10345) — appears in 7 questions
  (0, 1, 8489) — appears in 5 questions
  (0, 5, 10815) — appears in 4 questions


In [17]:
intervention_records = []

for q in QUESTIONS:
    qid = q['id']
    prompt = format_prompt(q, ACTIVE_TEMPLATE)
    input_ids = model.ensure_tokenized(prompt)

    original_logits, activations = model.get_activations(input_ids, sparse=True)
    last = original_logits.squeeze(0)[-1]
    margin_before = (last[IDX_TRUE] - last[IDX_FALSE]).item()

    active_set = {tuple(f.tolist()) for f in graphs[qid].active_features}
    cands = [f for f in recurring_features if tuple(f) in active_set]

    for interv_type, scale in [('amplify_5x', 5.0), ('ablate', 0.0)]:
        if cands:
            tuples = [
                (layer, pos, feat_idx, scale * activations[layer, pos, feat_idx])
                for (layer, pos, feat_idx) in cands
            ]
            interv_logits, _ = model.feature_intervention(input_ids, tuples)
        else:
            interv_logits = original_logits

        last_i = interv_logits.squeeze(0)[-1]
        margin_after = (last_i[IDX_TRUE] - last_i[IDX_FALSE]).item()
        delta = margin_after - margin_before

        intervention_records.append({
            'id': qid,
            'label': q['label'],
            'intervention': interv_type,
            'n_features': len(cands),
            'margin_before': round(margin_before, 4),
            'margin_after': round(margin_after, 4),
            'delta_margin': round(delta, 4),
            'predicted_before': margin_before > 0,
            'predicted_after': margin_after > 0,
            'flipped': (margin_before > 0) != (margin_after > 0),
        })

        cleanup_cuda()

interv_csv = RESULTS_DIR / 'intervention_results.csv'
with open(interv_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(intervention_records[0].keys()))
    writer.writeheader()
    writer.writerows(intervention_records)

print(f'Interventions complete. {len(intervention_records)} records saved to {interv_csv}')


Interventions complete. 16 records saved to /workspace/thesis_circuit_breaker/my_work/results/phase1/intervention_results.csv


---

## Section 6 — Results Summary

This section collects all results computed above into four consolidated tables.

**Table 1 — Baseline accuracy by template** shows how often the model predicted the correct label under each prompt format. A large gap between templates indicates that the model's classification is sensitive to surface phrasing, which is an important methodological finding.

**Table 2 — Baseline margins** shows the `True-False` logit margin for each question under the active template, giving a fine-grained picture of prediction strength and which questions were most confidently handled.

**Table 3 — Top recurring features** lists the features that appeared most frequently across the eight questions. These are the primary candidates for shared circuit components and the most informative targets for manual interpretability labelling.

**Table 4 — Intervention summary** reports the average change in `True-False` margin and the number of prediction flips caused by amplification and ablation. A consistent directional effect supports a causal interpretation of the recurring features.

In [18]:
print('=' * 55)
print('TABLE 1 — BASELINE ACCURACY BY TEMPLATE')
print('=' * 55)
for tkey in TEMPLATES:
    recs = [r for r in baseline_records if r['template'] == tkey]
    n_correct = sum(1 for r in recs if r['correct'])
    print(f'  {tkey}: {n_correct}/{len(recs)} correct ({100 * n_correct / len(recs):.0f}%)')

print()
print('=' * 55)
print(f'TABLE 2 — BASELINE MARGINS, TEMPLATE {ACTIVE_TEMPLATE}')
print('=' * 55)
print(f"{'ID':<8} {'Label':<6} {'Margin':>8}  Result")
print('-' * 32)
for r in [x for x in baseline_records if x['template'] == ACTIVE_TEMPLATE]:
    label_str = 'True ' if r['label'] else 'False'
    ok_str = 'OK' if r['correct'] else 'WRONG'
    print(f"{r['id']:<8} {label_str:<6} {r['margin']:>8.4f}  {ok_str}")

print()
print('=' * 55)
print('TABLE 3 — TOP RECURRING FEATURES')
print('=' * 55)
print(f"{'(layer, pos, feat_idx)':<32} {'Freq':>5}")
print('-' * 38)
for feat, qids in sorted_features[:10]:
    print(f'{str(feat):<32} {len(qids):>5}')

print()
print('=' * 55)
print('TABLE 4 — INTERVENTION SUMMARY')
print('=' * 55)
for interv_type in ['amplify_5x', 'ablate']:
    recs = [r for r in intervention_records if r['intervention'] == interv_type]
    if not recs:
        continue
    avg_delta = sum(r['delta_margin'] for r in recs) / len(recs)
    n_flipped = sum(1 for r in recs if r['flipped'])
    print(f'  {interv_type:<14}: avg delta_margin = {avg_delta:+.4f}   flipped: {n_flipped}/{len(recs)}')


TABLE 1 — BASELINE ACCURACY BY TEMPLATE
  T1: 4/8 correct (50%)
  T2: 4/8 correct (50%)
  T3: 4/8 correct (50%)

TABLE 2 — BASELINE MARGINS, TEMPLATE T1
ID       Label    Margin  Result
--------------------------------
TI-01    True     2.0000  OK
TI-02    False    1.8750  WRONG
TI-03    True     2.0000  OK
TI-04    False    2.0000  WRONG
TI-05    True     1.7500  OK
TI-06    False    1.7500  WRONG
TI-07    True     2.0000  OK
TI-08    False    2.2500  WRONG

TABLE 3 — TOP RECURRING FEATURES
(layer, pos, feat_idx)            Freq
--------------------------------------
(0, 2, 16200)                        8
(0, 1, 16229)                        8
(1, 1, 11660)                        8
(0, 1, 6083)                         8
(0, 1, 6421)                         8
(0, 1, 11167)                        8
(0, 1, 6381)                         7
(0, 1, 10345)                        7
(0, 1, 8489)                         5
(0, 5, 10815)                        4

TABLE 4 — INTERVENTION SUMMARY
  a

---

## Section 7 — Graph Export and Visualization

The raw attribution graphs stored as `.pt` files can be converted to the JSON format expected by the interactive graph viewer bundled with `circuit-tracer`. The viewer is based on the same frontend used in the Anthropic attribution graph papers and allows interactive exploration of feature nodes and edges.

We export graphs for the **most decisive true** and **most decisive false** questions — defined as the questions with the largest positive and most negative baseline margin respectively under the active template. These are the cases where the model's classification is clearest and where the graph is most likely to reveal a coherent, interpretable circuit.

After exporting, a local HTTP server is started and the viewer is embedded as an IFrame. Key interactions in the viewer are:

- **Click a node** to select it and see its feature visualisation.
- **Ctrl+click** to pin a node to the subgraph pane.
- **Hold G and click nodes** to group them into a labelled supernode.

On Runpod, the IFrame will not load automatically because the pod's localhost is not directly accessible. Enable port forwarding for the printed port number and open the URL in your local browser.

In [19]:
GRAPH_FILES_DIR = RESULTS_DIR / 'graph_files'
GRAPH_FILES_DIR.mkdir(parents=True, exist_ok=True)

t1_baseline_by_id = {r['id']: r for r in baseline_records if r['template'] == ACTIVE_TEMPLATE}

best_true = max(
    [q for q in QUESTIONS if q['label']],
    key=lambda q: t1_baseline_by_id[q['id']]['margin'],
)
best_false = max(
    [q for q in QUESTIONS if not q['label']],
    key=lambda q: -t1_baseline_by_id[q['id']]['margin'],
)

print('Exporting graphs for:')
print(f"  Best TRUE  item: {best_true['id']}  (margin={t1_baseline_by_id[best_true['id']]['margin']:+.4f})")
print(f"  Best FALSE item: {best_false['id']} (margin={t1_baseline_by_id[best_false['id']]['margin']:+.4f})")

for q in [best_true, best_false]:
    qid = q['id']
    graph_path = GRAPH_DIR / f'{qid}_{ACTIVE_TEMPLATE}.pt'
    slug = f'phase1-{qid.lower()}-{ACTIVE_TEMPLATE.lower()}'
    create_graph_files(
        graph_or_path=graph_path,
        slug=slug,
        output_path=str(GRAPH_FILES_DIR),
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f'  Graph files created: {slug}')


Exporting graphs for:
  Best TRUE  item: TI-01  (margin=+2.0000)
  Best FALSE item: TI-06 (margin=+1.7500)
  Graph files created: phase1-ti-01-t1
  Graph files created: phase1-ti-06-t1


In [20]:
PORT = 8047
server = serve(data_dir=str(GRAPH_FILES_DIR) + '/', port=PORT)

url = f'http://localhost:{PORT}/index.html'
print(f'Graph viewer running at: {url}')
print('On Runpod: enable port forwarding for this port and open the URL in your browser.')

display(IFrame(src=url, width='100%', height='800px'))


Graph viewer running at: http://localhost:8047/index.html
On Runpod: enable port forwarding for this port and open the URL in your browser.


In [ ]:
# server.stop()
